# FraudLens: exploratory data analysis

EDA only. Every reusable step lives in `src/fraudlens/`. This notebook loads data through the
same functions as the pipeline, so what you see here is what the model sees.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fraudlens.config import load_config
from fraudlens.data.loading import load_merged, memory_mb

cfg = load_config()
df = load_merged(cfg)  # ~1-2 GB RAM; pass nrows=100_000 for a quick look
print(f"{len(df):,} rows x {df.shape[1]} cols, {memory_mb(df):,.0f} MB")

## Class balance (why accuracy is meaningless here)

In [ ]:
rate = df["isFraud"].mean()
print(f"Fraud rate: {rate:.2%}, so 'approve everything' is {1 - rate:.2%} accurate")

## Fraud rate over time
`TransactionDT` is seconds from an undisclosed origin. Weekly fraud rate shows drift, which is why the split is chronological.

In [ ]:
week = (df["TransactionDT"] // (7 * 86_400)).rename("week")
weekly = df.groupby(week)["isFraud"].agg(["mean", "size"])
ax = weekly["mean"].plot(figsize=(8, 3), title="Weekly fraud rate")
ax.set_ylabel("fraud rate")
plt.show()

## Missingness
Columns above `features.max_missing_frac` (on the training block) are dropped by the pipeline.

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print((missing > cfg.features.max_missing_frac).sum(), "columns above the threshold on the full data")
missing.head(30)

## Amount by class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
for label, part in df.groupby("isFraud")["TransactionAmt"]:
    ax.hist(np.log1p(part), bins=80, alpha=0.6, density=True, label=["genuine", "fraud"][label])
ax.set_xlabel("log(1 + amount)")
ax.legend()
plt.show()

## Fraud rate by product and email domain

In [ ]:
display(df.groupby("ProductCD", observed=True)["isFraud"].agg(["mean", "size"]).sort_values("mean"))
display(
    df.groupby("P_emaildomain", observed=True)["isFraud"].agg(["mean", "size"])
    .query("size > 1000").sort_values("mean", ascending=False).head(10)
)

## Proxy card UID
How many transactions does each `card1_card2_card3_card5_addr1` key have?

In [ ]:
from fraudlens.features.behaviour import build_key

uid, _ = build_key(df, cfg.features.card_uid_cols)
sizes = uid.value_counts()
print(f"{len(sizes):,} proxy cards; median {sizes.median():.0f} txns; {(sizes == 1).mean():.0%} seen once")